### Use streamable http protocol for connecting to MCP server

In this demo, we will build an Agent using OpenAI SDK Agent framework.
We will leverage  streamable http protocol to connect Agent to MCP server.

We will use Uniprot MCP server for this. We will run this MCP server locally which will expose the server in localhost and then we will connect Agent to this MCP server via http protocol.

Steps
1. Download uniprot MCP server github
2. Install MCP server
3. Start the server
4. Build Agent with MCP connection
5. Interact with Agent with a chat interface



#### Uniprot MCP server
This is not official MCP server from uniprot

https://github.com/QuentinCody/uniprot-mcp-server


In [ ]:
# Clone the repo by 
# git clone https://github.com/QuentinCody/uniprot-mcp-server.git
# cd  uniprot-mcp-server

# install the server
# npm install

# Launch server. Opens at http://localhost:8787
# npm run dev

#### Import libraries

We will mainly import MCPServerStreamableHttp

In [ ]:
# Needed for loading environment variables from a .env file
from dotenv import load_dotenv

# Needed for defining agents and running the MCP server
from agents import Agent, Runner, trace, SQLiteSession
from agents.mcp import MCPServerStreamableHttp

# Needed for defining Local ollama model
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel

#### Load Environment

In [ ]:
# Load environment variables from .env file, overriding existing ones
load_dotenv(override=True)

#### Lets connect to Uniprot MCP server
Check available tools in the MCP server

The server is running at http://localhost:8787

The url for mcp would be http://localhost:8787/mcp

![alt text](image.png)

In [ ]:
# https://github.com/QuentinCody/uniprot-mcp-server
# This with block connects to the MCP server with an object called server
# The parameters define the URL of the server and various timeout and retry settings
# Inside the with block, we call server.list_tools() to get the list of tools available
# on the MCP server. This is done asynchronously using await.
# Finally, we print the list of tools retrieved from the server.
async with MCPServerStreamableHttp(params = {
            "url": "http://localhost:8787/mcp",
            "timeout": 60,
            "sse_read_timeout": 300,
        }, 
        max_retry_attempts=2,
        retry_backoff_seconds_base=2.0,
        client_session_timeout_seconds=60,
        cache_tools_list= True
        ) as server:
    tool_lists = await server.list_tools()

tool_lists

#### Build Agent with Uniprot MCP server

In [ ]:
# This is the openai gpt model. This is paid model.
# model = "gpt-4.1-nano"


# Alternatively you can ue local ollama model which is free
# Make sure you have ollama installed and the model downloaded
# https://ollama.com/docs/installation
# ollama pull llama3.2:latest
# To use reasoning capabilities, use the gpt-oss model
# ollama pull gpt-oss
# In my system ollama server runs at http://localhost:11434

client = AsyncOpenAI(base_url="http://localhost:11434/v1")
# model_name = "llama3.2:latest"
model_name = "gpt-oss"
model = OpenAIChatCompletionsModel(model = model_name,openai_client= client)

# The system prompt for the agent
agent_instructions = (
        "You are a expert in Uniprot"
    )

# Create a session to store conversation history
session = SQLiteSession("uniprot_session")

# Define the chat function
# This function takes a message and history as input
# It connects to the MCP server and creates an agent with the given instructions and model
# It then runs the agent with the input message and returns the final output
async def chat(message, history):
    async with MCPServerStreamableHttp(params = {
                "url": "http://localhost:8787/mcp",
                "timeout": 60,
                "sse_read_timeout": 300,
            }, 
            max_retry_attempts=2,
            retry_backoff_seconds_base=2.0,
            client_session_timeout_seconds=60,
            cache_tools_list= True
            ) as server:
        uniprot_agent = Agent(name="uniprot_agent",
                        instructions=agent_instructions,
                        model=model,
                        mcp_servers=[server])
        result = await Runner.run(uniprot_agent, message)
        return result.final_output

# Some example to try
# get uniprot ids for gene TP53. just 5 is enough. only uniprot ids
# Next ask describe in 2 lines about P04637

# Create a Gradio chat interface
import gradio as gr
gr.ChatInterface(
    chat,
    title="Uniprot Chatbot",
    description="Chat with the Uniprot Expert"
).launch() 